In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def build_prompt(tweet):
    example_tweet = "People are being evacuated as the storm surges intensify along the coast."
    example_disaster = "hurricane"
    example_human = "displaced_people_and_evacuations"

    return f"""
<|system|>
You are an expert disaster response analyst.
For each tweet, you must:
1. Think step-by-step internally about what disaster it refers to.
2. Think step-by-step internally about what humanitarian category it expresses.
3. Do NOT reveal your reasoning.
4. Output ONLY the final JSON.

Below are the allowed label sets:

Allowed Disaster Types:
{', '.join(disaster_encoder.classes_)}

Allowed Humanitarian Categories:
{', '.join(human_encoder.classes_)}

Here is an example to guide you.

<|user|>
Tweet: "{example_tweet}"

Classify the disaster type and humanitarian category.
Think step-by-step internally, then output the final JSON only.

<|assistant|>
Final Answer:
{{
  "disaster_type": "{example_disaster}",
  "humanitarian_category": "{example_human}"
}}

<|user|>
Now classify the following tweet in the same way:

Tweet: "{tweet}"

Remember:
- Think step-by-step internally.
- Do NOT show your reasoning.
- Only output the final JSON object:

{{
  "disaster_type": "...",
  "humanitarian_category": "..."
}}
"""

In [ ]:
def build_prompt(tweet):
    system = (
        "You are an expert disaster response analyst. Think step-by-step before giving the final answer.\n"
        "You will be given a tweet. First reason internally about which disaster it most likely refers to "
        "and what humanitarian category it belongs to. After completing your reasoning, output ONLY the final JSON.\n"
        "Do NOT reveal your reasoning in the final JSON response.\n"
    )

    user = (
        f"Tweet: \"{tweet}\"\n\n"
        "Possible disaster types: " + ", ".join(disaster_encoder.classes_) + "\n"
        "Possible humanitarian categories: " + ", ".join(human_encoder.classes_) + "\n\n"
        "Follow these steps:\n"
        "1. Think step-by-step about the content of the tweet (internally).\n"
        "2. Decide the most likely disaster type.\n"
        "3. Decide the most likely humanitarian category.\n"
        "4. DO NOT show your reasoning.\n"
        "5. Output ONLY the final JSON exactly in this format:\n"
        "{\n"
        "  \"disaster_type\": \"...\",\n"
        "  \"humanitarian_category\": \"...\"\n"
        "}\n"
    )

    full_prompt = f"<|system|>\n{system}\n<|user|>\n{user}\n<|assistant|>\n"

    return full_prompt

In [ ]:
# Qwen Prompt-Based Multi-Class Disaster Tweet Classification (humAID)
import pandas as pd
import torch
import os
import json
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

!pip install --force-reinstall bitsandbytes

def read_messy_tsv(file_path):
    with open(file_path, 'r', encoding='utf-8-sig', errors='replace') as f:
        lines = f.readlines()

    if not lines:
        return pd.DataFrame(columns=['tweet_text', 'class_label', 'disaster_type'])

    delim = None
    for l in lines:
        if '\t' in l:
            delim = '\t'
            break
        if ',' in l:
            delim = ','
            break
    if delim is None:
        # very unlikely, but fall back to comma
        delim = ','

    header_idx = -1
    for i, l in enumerate(lines):
        low = l.lower()
        if 'tweet_id' in low and 'tweet_text' in low and 'class_label' in low:
            header_idx = i
            break

    # start reading after header if found, otherwise from top
    start = header_idx + 1 if header_idx != -1 else 0

    processed_data = []

    for line in lines[start:]:
        line = line.strip('\r\n')
        if not line:
            continue

        parts = line.split(delim)
        # We expect at least: tweet_id | tweet_text | class_label | disaster
        if len(parts) < 3:
            continue

        # last column: disaster
        disaster = parts[-1].strip().strip('"').strip("'")
        # second last: class_label
        class_label = parts[-2].strip().strip('"').strip("'")
        # everything between first and second-last: tweet_text (handles extra commas/tabs)
        middle = parts[1:-2]
        tweet_text = delim.join(middle).strip().strip('"').strip("'")

        if tweet_text == '' or class_label == '' or disaster == '':
            continue

        processed_data.append([tweet_text, class_label, disaster])

    if not processed_data:
        return pd.DataFrame(columns=['tweet_text', 'class_label', 'disaster_type'])

    df = pd.DataFrame(processed_data,
                      columns=['tweet_text', 'class_label', 'disaster_type'])
    df.reset_index(drop=True, inplace=True)
    if 'disaster_type' in df.columns and 'disaster' not in df.columns:
        df['disaster'] = df['disaster_type']
    elif 'disaster' in df.columns and 'disaster_type' not in df.columns:
        df['disaster_type'] = df['disaster']
    return df

data_folder = "/content/drive/MyDrive/humAID_dataset_new/"

def load_all_splits(base_folder):
    train_list, dev_list, test_list = [], [], []
    folder_count = 0
    for subfolder in os.listdir(base_folder):
        subpath = os.path.join(base_folder, subfolder)
        if not os.path.isdir(subpath):
            continue

        folder_count += 1
        print(f"Processing folder: {subfolder}") # Added print statement for directory name

        current_subfolder_train_count = 0
        current_subfolder_dev_count = 0
        current_subfolder_test_count = 0

        for split in ['train', 'dev', 'test']:
            file_path = os.path.join(subpath, f"{subfolder}_{split}.tsv")
            if os.path.exists(file_path):
                df = read_messy_tsv(file_path)
                if split == 'train':
                    train_list.append(df)
                    current_subfolder_train_count = len(df)
                elif split == 'dev':
                    dev_list.append(df)
                    current_subfolder_dev_count = len(df)
                else:
                    test_list.append(df)
                    current_subfolder_test_count = len(df)

        print(f"  {subfolder} counts: Train={current_subfolder_train_count}, Dev={current_subfolder_dev_count}, Test={current_subfolder_test_count}")

    train_df = pd.concat(train_list, ignore_index=True) if train_list else pd.DataFrame()
    dev_df = pd.concat(dev_list, ignore_index=True) if dev_list else pd.DataFrame()
    test_df = pd.concat(test_list, ignore_index=True) if test_list else pd.DataFrame()

    print(f"Processed {folder_count} folders from {base_folder}.")
    print(f" Loaded: {len(train_df)} train, {len(dev_df)} dev, {len(test_df)} test samples.")
    return train_df, dev_df, test_df

train_df, dev_df, test_df = load_all_splits(data_folder)

for df in [train_df, dev_df, test_df]:
    df.rename(columns={'tweet_text': 'text', 'class_label': 'text_humanitarian'}, inplace=True)
    df.dropna(subset=['text', 'text_humanitarian', 'disaster_type'], inplace=True)

train_df = pd.concat([train_df, dev_df], ignore_index=True)

train_df['disaster_norm'] = train_df['disaster_type'].str.lower()
train_df['human_norm'] = train_df['text_humanitarian'].str.lower()

test_df['disaster_norm'] = test_df['disaster_type'].str.lower()
test_df['human_norm'] = test_df['text_humanitarian'].str.lower()

# Fit encoders **on normalized labels only**
disaster_encoder = LabelEncoder()
human_encoder = LabelEncoder()

train_df['disaster_label'] = disaster_encoder.fit_transform(train_df['disaster_norm'])
train_df['human_label'] = human_encoder.fit_transform(train_df['human_norm'])

# map only normalized labels in test set
test_df['disaster_label'] = test_df['disaster_norm'].map(
    lambda x: disaster_encoder.transform([x])[0] if x in disaster_encoder.classes_ else -1
)
test_df['human_label'] = test_df['human_norm'].map(
    lambda x: human_encoder.transform([x])[0] if x in human_encoder.classes_ else -1
)

test_df = test_df[(test_df['disaster_label'] != -1) & (test_df['human_label'] != -1)]

print("\nNormalized Disaster types:", list(disaster_encoder.classes_))
print("Normalized Humanitarian types:", list(human_encoder.classes_))

model_name = "Qwen/Qwen2.5-7B-Instruct"

# Clear any previously loaded models from GPU memory if they exist
if 'model' in globals() and isinstance(model, torch.nn.Module):
    print("Clearing previously loaded PyTorch model from GPU memory...")
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Done clearing GPU memory.")

# Detect device
if torch.cuda.is_available():
    device_map = "auto"
    compute_dtype = torch.bfloat16 # Use bfloat16 for compute_dtype with 4-bit quantization
else:
    device_map = "cpu"
    compute_dtype = torch.float32 # Use float32 on CPU

print(f"Loading {model_name} with memory optimization...")

# Clear CUDA cache right before loading this model to ensure maximum free memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Configure 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Try lightweight loading with automatic offload and 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map=device_map,
    # Use 'dtype' instead of 'torch_dtype' and align with compute_dtype
    dtype=compute_dtype if torch.cuda.is_available() else torch.float32,
    low_cpu_mem_usage=True,
    quantization_config=quantization_config # Apply quantization
)

# Create generator pipeline (no full model copy)
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150,
    temperature=0.2,
    do_sample=False
)

print("Model loaded successfully with optimized memory settings.")

def build_prompt(tweet):
    allowed_disasters = ", ".join(disaster_encoder.classes_)
    allowed_humans = ", ".join(human_encoder.classes_)

    return f"""
You are an expert disaster response analyst.

You will be given a tweet about a disaster.
Your tasks are:
1. Identify the *disaster type*.
2. Identify the *humanitarian category*.

Below are the allowed label sets.

Allowed Disaster Types:
{allowed_disasters}

Allowed Humanitarian Categories:
{allowed_humans}

---

### Example 1
...
(FULL SAME EXAMPLE TEXT IF YOU WANT)
...

Now classify the following tweet in the **same way**:

Tweet:
"{tweet}"

### Your Task

1. First, write your reasoning in natural language.
2. Then output ONLY the final JSON in this exact format:

Final Answer:
{{
  "disaster_type": "ONE_LABEL_FROM_Disaster_Types",
  "humanitarian_category": "ONE_LABEL_FROM_Humanitarian_Categories"
}}

Do NOT use angle brackets <> in the JSON.
Do NOT invent new labels; choose exactly one from each allowed list.
"""

import os
import json
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score

def run_inference_and_save(generator, test_df, disaster_encoder, human_encoder, save_path, n_samples=10, random_state=42):
    """
    Runs prompt-based inference on n_samples from test_df, appends results to CSV,
    and evaluates on the cumulative data (all runs so far).
    """

    # --- Load existing results if present ---
    if os.path.exists(save_path):
        results_df = pd.read_csv(save_path)
        done_texts = set(results_df["text"].tolist())
        print(f"Found existing results: {len(results_df)} samples already processed.")
    else:
        results_df = pd.DataFrame(columns=["text", "true_disaster", "pred_disaster", "true_human", "pred_human"])
        done_texts = set()
        print("No existing results found. Starting fresh.")

    # --- Pick new samples not yet processed ---
    remaining_df = test_df[~test_df["text"].isin(done_texts)]
    if len(remaining_df) == 0:
        print("All samples already processed!")
        return

    subset = remaining_df.sample(n=min(n_samples, len(remaining_df)), random_state=random_state)
    print(f"Running inference on {len(subset)} new samples...")

    import re

    # --- Run inference for selected samples ---
    new_results = []
    for _, row in subset.iterrows():
        tweet = row['text']
        true_disaster = row['disaster_type']
        true_human = row['text_humanitarian']

        prompt = build_prompt(tweet)
        response = generator(prompt, max_new_tokens=150, temperature=0.2, do_sample=False)[0]['generated_text']

        # Only look after the last "Final Answer:"
        if "Final Answer:" in response:
            final_part = response.split("Final Answer:", 1)[-1]
        else:
            final_part = response

        matches = re.findall(r'\{.*?\}', final_part, re.DOTALL)
        parsed = None
        if matches:
            json_part = matches[-1]
            json_part = json_part.replace("“", "\"").replace("”", "\"").replace("'", "\"")
            json_part = re.sub(r",\s*\}", "}", json_part)
            try:
                parsed = json.loads(json_part)
            except:
                parsed = None

        if parsed is not None:
            pred_disaster = parsed.get("disaster_type", "").strip()
            pred_human = parsed.get("humanitarian_category", "").strip()
        else:
            pred_disaster, pred_human = "unknown", "unknown"


        new_results.append({
            "text": tweet,
            "true_disaster": true_disaster,
            "pred_disaster": pred_disaster,
            "true_human": true_human,
            "pred_human": pred_human
        })

    # --- Append new results to existing CSV ---
    new_df = pd.DataFrame(new_results)
    combined_df = pd.concat([results_df, new_df], ignore_index=True)
    combined_df.to_csv(save_path, index=False)
    print(f"\nSaved cumulative results to: {save_path} (Total {len(combined_df)} samples)")

    # --- Compute cumulative evaluation ---
    true_disaster_all = combined_df["true_disaster"].tolist()
    pred_disaster_all = combined_df["pred_disaster"].tolist()
    true_human_all = combined_df["true_human"].tolist()
    pred_human_all = combined_df["pred_human"].tolist()

    print("\n===== Disaster Type Classification (Cumulative) =====")
    print(classification_report(true_disaster_all, pred_disaster_all, labels=disaster_encoder.classes_))
    print(f"Overall Disaster Type Accuracy: {accuracy_score(true_disaster_all, pred_disaster_all):.4f}")

    print("\n===== Humanitarian Type Classification (Cumulative) =====")
    print(classification_report(true_human_all, pred_human_all, labels=human_encoder.classes_))
    print(f"Overall Humanitarian Type Accuracy: {accuracy_score(true_human_all, pred_human_all):.4f}")

save_path = "/content/drive/MyDrive/humAID_Qwen_Prompt_Results_cot.csv"

# Run for first 10 samples
run_inference_and_save(generator, test_df, disaster_encoder, human_encoder, save_path, n_samples=50)

# Later, you can run more in new cells:
# run_inference_and_save(generator, test_df, disaster_encoder, human_encoder, save_path, n_samples=50)


  Using cached bitsandbytes-0.48.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached torch-2.9.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (30 kB)
  Using cached numpy-2.3.5-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached packaging-25.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2025.12.0-py3-none-any.whl.metadata (10 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64

Processing folder: canade_wildfires_2016
  canade_wildfires_2016 counts: Train=0, Dev=0, Test=0
Processing folder: hurricane_matthew_2016
  hurricane_matthew_2016 counts: Train=1157, Dev=168, Test=329
Processing folder: srilanka_floods_2017
  srilanka_floods_2017 counts: Train=392, Dev=57, Test=111
Processing folder: pakistan_earthquake_2019
  pakistan_earthquake_2019 counts: Train=1370, Dev=199, Test=389
Processing folder: california_wildfires_2018
  california_wildfires_2018 counts: Train=5163, Dev=752, Test=1461
Processing folder: greece_wildfires_2018
  greece_wildfires_2018 counts: Train=1060, Dev=154, Test=301
Processing folder: italy_earthquake_aug_2016
  italy_earthquake_aug_2016 counts: Train=840, Dev=122, Test=239
Processing folder: cyclone_idai_2019
  cyclone_idai_2019 counts: Train=2753, Dev=401, Test=779
Processing folder: hurricane_dorian_2019
  hurricane_dorian_2019 counts: Train=5329, Dev=776, Test=1508
Processing folder: ecuador_earthquake_2016
  ecuador_earthquake_201

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Model loaded successfully with optimized memory settings.
🆕 No existing results found. Starting fresh.
🧪 Running inference on 50 new samples...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



💾 Saved cumulative results to: /content/drive/MyDrive/humAID_Qwen_Prompt_Results_cot.csv (Total 50 samples)

===== Disaster Type Classification (Cumulative) =====
              precision    recall  f1-score   support

     cyclone       0.00      0.00      0.00         0
  earthquake       0.40      1.00      0.57         2
       flood       0.00      0.00      0.00         0
      floods       1.00      0.25      0.40         4
   hurricane       0.35      0.70      0.47        10
    wildfire       0.67      1.00      0.80         2

   micro avg       0.32      0.67      0.44        18
   macro avg       0.40      0.49      0.37        18
weighted avg       0.54      0.67      0.50        18

🎯 Overall Disaster Type Accuracy: 0.2400

===== Humanitarian Type Classification (Cumulative) =====
                                        precision    recall  f1-score   support

                    caution_and_advice       0.00      0.00      0.00         4
      displaced_people_and_evacu

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.p

In [ ]:
!pip install -U bitsandbytes

In [ ]:
# Qwen Prompt-Based Multi-Class Disaster Tweet Classification (humAID)
!pip install --force-reinstall -q bitsandbytes

import os
import json
import numpy as np
import pandas as pd
import torch
import re

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_recall_fscore_support,
)

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    BitsAndBytesConfig,
)

def read_messy_tsv(file_path):
    with open(file_path, 'r', encoding='utf-8-sig', errors='replace') as f:
        lines = f.readlines()

    if not lines:
        return pd.DataFrame(columns=['tweet_text', 'class_label', 'disaster_type'])

    # -------- 1) Detect delimiter for this file (tab or comma) --------
    delim = None
    for l in lines:
        if '\t' in l:
            delim = '\t'
            break
        if ',' in l:
            delim = ','
            break
    if delim is None:
        delim = ','

    # -------- 2) Find header row (where tweet_id, tweet_text, class_label appear) --------
    header_idx = -1
    for i, l in enumerate(lines):
        low = l.lower()
        if 'tweet_id' in low and 'tweet_text' in low and 'class_label' in low:
            header_idx = i
            break

    # start reading after header if found, otherwise from top
    start = header_idx + 1 if header_idx != -1 else 0

    processed_data = []

    # -------- 3) Parse each data row robustly --------
    for line in lines[start:]:
        line = line.strip('\r\n')
        if not line:
            continue

        parts = line.split(delim)
        # We expect at least: tweet_id | tweet_text | class_label | disaster
        if len(parts) < 3:
            continue

        # last column: disaster
        disaster = parts[-1].strip().strip('"').strip("'")
        # second last: class_label
        class_label = parts[-2].strip().strip('"').strip("'")
        # everything between first and second-last: tweet_text (handles extra commas/tabs)
        middle = parts[1:-2]
        tweet_text = delim.join(middle).strip().strip('"').strip("'")

        if tweet_text == '' or class_label == '' or disaster == '':
            continue

        processed_data.append([tweet_text, class_label, disaster])

    if not processed_data:
        return pd.DataFrame(columns=['tweet_text', 'class_label', 'disaster_type'])

    df = pd.DataFrame(processed_data,
                      columns=['tweet_text', 'class_label', 'disaster_type'])
    df.reset_index(drop=True, inplace=True)
    return df


def parse_event_metadata(subfolder_name):
    """
    Parse event_location and event_year from folder name.
    Example: 'california_wildfires_2018' -> ('california_wildfires', '2018')
    """
    parts = subfolder_name.split("_")
    if len(parts) >= 2 and parts[-1].isdigit() and len(parts[-1]) == 4:
        year = parts[-1]
        location_tag = "_".join(parts[:-1])
    else:
        year = "unknown"
        location_tag = subfolder_name
    return location_tag, year


# Path to your humAID dataset on Drive
data_folder = "/content/drive/MyDrive/humAID_dataset_new/"

def load_all_splits(base_folder):
    train_list, dev_list, test_list = [], [], []
    folder_count = 0

    for subfolder in os.listdir(base_folder):
        subpath = os.path.join(base_folder, subfolder)
        if not os.path.isdir(subpath):
            continue

        folder_count += 1
        print(f"Processing folder: {subfolder}")

        event_location, event_year = parse_event_metadata(subfolder)

        current_subfolder_train_count = 0
        current_subfolder_dev_count = 0
        current_subfolder_test_count = 0

        for split in ['train', 'dev', 'test']:
            file_path = os.path.join(subpath, f"{subfolder}_{split}.tsv")
            if os.path.exists(file_path):
                df = read_messy_tsv(file_path)

                # attach event metadata
                df["event_location"] = event_location
                df["event_year"] = event_year

                if split == 'train':
                    train_list.append(df)
                    current_subfolder_train_count = len(df)
                elif split == 'dev':
                    dev_list.append(df)
                    current_subfolder_dev_count = len(df)
                else:
                    test_list.append(df)
                    current_subfolder_test_count = len(df)

        print(
            f"  {subfolder} counts: "
            f"Train={current_subfolder_train_count}, "
            f"Dev={current_subfolder_dev_count}, "
            f"Test={current_subfolder_test_count}"
        )

    train_df = pd.concat(train_list, ignore_index=True) if train_list else pd.DataFrame()
    dev_df = pd.concat(dev_list, ignore_index=True) if dev_list else pd.DataFrame()
    test_df = pd.concat(test_list, ignore_index=True) if test_list else pd.DataFrame()

    print(f"Processed {folder_count} folders from {base_folder}.")
    print(f" Loaded: {len(train_df)} train, {len(dev_df)} dev, {len(test_df)} test samples.")
    return train_df, dev_df, test_df


train_df, dev_df, test_df = load_all_splits(data_folder)

# rename text columns + drop NaNs (location/year columns are kept)
for df in [train_df, dev_df, test_df]:
    df.rename(columns={'tweet_text': 'text', 'class_label': 'text_humanitarian'}, inplace=True)
    df.dropna(subset=['text', 'text_humanitarian', 'disaster_type'], inplace=True)

# merge train + dev for label fitting
train_df = pd.concat([train_df, dev_df], ignore_index=True)

disaster_encoder = LabelEncoder()
human_encoder = LabelEncoder()
location_encoder = LabelEncoder()
year_encoder = LabelEncoder()

train_df['disaster_label'] = disaster_encoder.fit_transform(train_df['disaster_type'])
train_df['human_label'] = human_encoder.fit_transform(train_df['text_humanitarian'])
train_df['location_label'] = location_encoder.fit_transform(train_df['event_location'])
train_df['year_label'] = year_encoder.fit_transform(train_df['event_year'])

# map test labels; unknown -> -1
test_df['disaster_label'] = test_df['disaster_type'].map(
    lambda x: disaster_encoder.transform([x])[0] if x in disaster_encoder.classes_ else -1
)
test_df['human_label'] = test_df['text_humanitarian'].map(
    lambda x: human_encoder.transform([x])[0] if x in human_encoder.classes_ else -1
)
test_df['location_label'] = test_df['event_location'].map(
    lambda x: location_encoder.transform([x])[0] if x in location_encoder.classes_ else -1
)
test_df['year_label'] = test_df['event_year'].map(
    lambda x: year_encoder.transform([x])[0] if x in year_encoder.classes_ else -1
)

test_df = test_df[
    (test_df['disaster_label'] != -1) &
    (test_df['human_label'] != -1) &
    (test_df['location_label'] != -1) &
    (test_df['year_label'] != -1)
]

print("\nDisaster types:", list(disaster_encoder.classes_))
print("Humanitarian types:", list(human_encoder.classes_))
print("Event locations:", list(location_encoder.classes_))
print("Event years:", list(year_encoder.classes_))

model_name = "Qwen/Qwen2.5-7B-Instruct"

# Clear any previously loaded models from GPU memory if they exist
if 'model' in globals() and isinstance(model, torch.nn.Module):
    print("Clearing previously loaded PyTorch model from GPU memory...")
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Done clearing GPU memory.")

# Detect device
if torch.cuda.is_available():
    device_map = "auto"
    compute_dtype = torch.bfloat16  # Use bfloat16 for compute_dtype with 4-bit quantization
else:
    device_map = "cpu"
    compute_dtype = torch.float32   # Use float32 on CPU

print(f"\n⚙️ Loading {model_name} with memory optimization...")

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Configure 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer + model
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map=device_map,
    dtype=compute_dtype if torch.cuda.is_available() else torch.float32,
    low_cpu_mem_usage=True,
    quantization_config=quantization_config,
)

# Create generator pipeline
# (don't pass temperature/top_p here to avoid extra warnings)
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,  # a bit more room for reasoning + JSON
    do_sample=False,
)

print("Model loaded successfully with optimized memory settings.")

def build_prompt(tweet: str) -> str:
    """
    CoT-style prompt, but the JSON section is written in terms of the actual labels,
    not placeholders with <...>. This makes it more likely Qwen will output a valid
    label string we can compare to the ground truth.
    """
    allowed_disasters = ", ".join(disaster_encoder.classes_)
    allowed_humans = ", ".join(human_encoder.classes_)
    allowed_locs = ", ".join(location_encoder.classes_)
    allowed_years = ", ".join(year_encoder.classes_)

    return f"""
You are an expert disaster response analyst.

You will be given a tweet about a disaster.

Your tasks are:
1. Identify the *disaster type*.
2. Identify the *humanitarian category*.
3. Identify the *event location label* (from the allowed list).
4. Identify the *event year* (from the allowed list).

Below are the allowed label sets (you MUST choose only from these):

Allowed Disaster Types:
{allowed_disasters}

Allowed Humanitarian Categories:
{allowed_humans}

Allowed Event Locations:
{allowed_locs}

Allowed Event Years:
{allowed_years}

---

### Example 1

Tweet:
"People are being evacuated as the storm surges intensify along the coast and authorities warn of dangerous winds."

Reasoning (example):
- The tweet mentions storm surges and dangerous winds, typical of a tropical cyclone/hurricane.
- Evacuations indicate displaced people.
- So pick one disaster type from the allowed list and one humanitarian category about displacement.
- Choose the most plausible event_location and event_year from the allowed sets.

Final Answer (example):
{{
  "disaster_type": "{disaster_encoder.classes_[0]}",
  "humanitarian_category": "{human_encoder.classes_[0]}",
  "event_location": "{location_encoder.classes_[0]}",
  "event_year": "{year_encoder.classes_[0]}"
}}

---

Now classify the following tweet in the same way.

Tweet:
"{tweet}"

### Your Task

1. First, think step by step in natural language about:
   - What disaster is being described?
   - What textual clues indicate this disaster type?
   - What humanitarian impact or need is mentioned?
   - Which event_location label from the allowed list is most plausible?
   - Which event_year from the allowed list is most plausible?

2. Then, at the end, output ONLY the final JSON in this exact format:

Final Answer:
{{
  "disaster_type": "ONE_LABEL_FROM_Disaster_Types",
  "humanitarian_category": "ONE_LABEL_FROM_Humanitarian_Categories",
  "event_location": "ONE_LABEL_FROM_Event_Locations",
  "event_year": "ONE_LABEL_FROM_Event_Years"
}}
"""

def run_inference_and_save(
    generator,
    test_df,
    disaster_encoder,
    human_encoder,
    location_encoder,
    year_encoder,
    save_path,
    n_samples=10,
    random_state=42,
):
    """
    Runs prompt-based inference on n_samples from test_df, appends results to CSV,
    and evaluates on the cumulative data (all runs so far).
    """

    # --- Load existing results if present ---
    if os.path.exists(save_path):
        results_df = pd.read_csv(save_path)
        done_texts = set(results_df["text"].tolist())
        print(f"Found existing results: {len(results_df)} samples already processed.")
    else:
        cols = [
            "text",
            "true_disaster", "pred_disaster",
            "true_human", "pred_human",
            "true_location", "pred_location",
            "true_year", "pred_year",
        ]
        results_df = pd.DataFrame(columns=cols)
        done_texts = set()
        print("No existing results found. Starting fresh.")

    # --- Pick new samples not yet processed ---
    remaining_df = test_df[~test_df["text"].isin(done_texts)]
    if len(remaining_df) == 0:
        print("All samples already processed!")
        combined_df = results_df
    else:
        subset = remaining_df.sample(n=min(n_samples, len(remaining_df)), random_state=random_state)
        print(f"Running inference on {len(subset)} new samples...")

        new_results = []
        for i, (_, row) in enumerate(subset.iterrows()):
            tweet = row['text']
            true_disaster = row['disaster_type']
            true_human = row['text_humanitarian']
            true_loc = row['event_location']
            true_year = row['event_year']

            prompt = build_prompt(tweet)
            # Generate full text (prompt echo + reasoning + Final Answer)
            out = generator(prompt, max_new_tokens=200)[0]['generated_text']

            # For sanity: show the first couple of samples' outputs
            if i < 3:
                print("\n========== SAMPLE DEBUG ==========")
                print("Tweet:", tweet)
                print("Model raw (first 600 chars):", out[:600].replace("\n", " ")[:600], "...")
                print("==================================")

            # --- Extract the "Final Answer" block ---
            if "Final Answer:" in out:
                final_block = out.split("Final Answer:", 1)[-1].strip()
            else:
                final_block = out  # fallback

            # Find the last JSON-looking block in the final_block
            matches = re.findall(r'\{.*?\}', final_block, re.DOTALL)
            parsed = None
            if matches:
                json_part = matches[-1]
                json_part = json_part.replace("“", "\"").replace("”", "\"").replace("'", "\"")
                json_part = re.sub(r",\s*\}", "}", json_part)  # trailing commas
                try:
                    parsed = json.loads(json_part)
                except Exception:
                    parsed = None

            if parsed is not None:
                pred_disaster = parsed.get("disaster_type", "").strip()
                pred_human = parsed.get("humanitarian_category", "").strip()
                pred_loc = parsed.get("event_location", "").strip()
                pred_year = parsed.get("event_year", "").strip()
            else:
                pred_disaster = pred_human = pred_loc = pred_year = "unknown"

            # Append one row of results
            new_results.append({
                "text": tweet,
                "true_disaster": true_disaster,
                "pred_disaster": pred_disaster,
                "true_human": true_human,
                "pred_human": pred_human,
                "true_location": true_loc,
                "pred_location": pred_loc,
                "true_year": true_year,
                "pred_year": pred_year,
            })

        # Merge with previous results and save
        new_df = pd.DataFrame(new_results)
        combined_df = pd.concat([results_df, new_df], ignore_index=True)
        combined_df.to_csv(save_path, index=False)
        print(f"\nSaved cumulative results to: {save_path} (Total {len(combined_df)} samples)")

    # --- Compute cumulative evaluation ---
    true_disaster_all = combined_df["true_disaster"].tolist()
    pred_disaster_all = combined_df["pred_disaster"].tolist()
    true_human_all = combined_df["true_human"].tolist()
    pred_human_all = combined_df["pred_human"].tolist()
    true_loc_all = combined_df["true_location"].tolist()
    pred_loc_all = combined_df["pred_location"].tolist()
    true_year_all = combined_df["true_year"].tolist()
    pred_year_all = combined_df["pred_year"].tolist()

    print("\n===== Disaster Type Classification (Cumulative) =====")
    print(classification_report(true_disaster_all, pred_disaster_all,
                                labels=list(disaster_encoder.classes_),
                                zero_division=0))
    print(f"Overall Disaster Type Accuracy: {accuracy_score(true_disaster_all, pred_disaster_all):.4f}")

    print("\n===== Humanitarian Type Classification (Cumulative) =====")
    print(classification_report(true_human_all, pred_human_all,
                                labels=list(human_encoder.classes_),
                                zero_division=0))
    print(f"Overall Humanitarian Type Accuracy: {accuracy_score(true_human_all, pred_human_all):.4f}")

    print("\n===== Event Location Classification (Cumulative) =====")
    print(classification_report(true_loc_all, pred_loc_all,
                                labels=list(location_encoder.classes_),
                                zero_division=0))
    print(f"Overall Location Accuracy: {accuracy_score(true_loc_all, pred_loc_all):.4f}")

    print("\n===== Event Year Classification (Cumulative) =====")
    print(classification_report(true_year_all, pred_year_all,
                                labels=list(year_encoder.classes_),
                                zero_division=0))
    print(f"Overall Year Accuracy: {accuracy_score(true_year_all, pred_year_all):.4f}")

    # Compact macro summary for table
    d_prec, d_rec, d_f1, _ = precision_recall_fscore_support(true_disaster_all, pred_disaster_all, average="macro", zero_division=0)
    h_prec, h_rec, h_f1, _ = precision_recall_fscore_support(true_human_all, pred_human_all, average="macro", zero_division=0)
    l_prec, l_rec, l_f1, _ = precision_recall_fscore_support(true_loc_all, pred_loc_all, average="macro", zero_division=0)
    y_prec, y_rec, y_f1, _ = precision_recall_fscore_support(true_year_all, pred_year_all, average="macro", zero_division=0)

    print("\n===== Macro Summary (Qwen-COT-pretrained) =====")
    print(f"Disaster   : Acc={accuracy_score(true_disaster_all, pred_disaster_all):.3f}, P={d_prec:.3f}, R={d_rec:.3f}, F1={d_f1:.3f}")
    print(f"Humanit.   : Acc={accuracy_score(true_human_all, pred_human_all):.3f}, P={h_prec:.3f}, R={h_rec:.3f}, F1={h_f1:.3f}")
    print(f"Location   : Acc={accuracy_score(true_loc_all, pred_loc_all):.3f}, P={l_prec:.3f}, R={l_rec:.3f}, F1={l_f1:.3f}")
    print(f"Year       : Acc={accuracy_score(true_year_all, pred_year_all):.3f}, P={y_prec:.3f}, R={y_rec:.3f}, F1={y_f1:.3f}")

save_path = "/content/drive/MyDrive/humAID_Qwen_Prompt_Results_new_dataset_cot.csv"

run_inference_and_save(
    generator,
    test_df,
    disaster_encoder,
    human_encoder,
    location_encoder,
    year_encoder,
    save_path,
    n_samples=50,
)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 837.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 126.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━ 342.7/706.8 MB 3.4 MB/s eta 0:01:47
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


✅ Model loaded successfully with optimized memory settings.
📂 Found existing results: 100 samples already processed.
🧪 Running inference on 50 new samples...

========== SAMPLE DEBUG ==========
Tweet: Thank you NY Times Opinion Columnist David Brooks for this great look at rural life in Nebraska, including bonus mention of MNB Bank in McCook, headed by Mark Graff:
Model raw (first 600 chars):  You are an expert disaster response analyst.  You will be given a tweet about a disaster.  Your tasks are: 1. Identify the *disaster type*. 2. Identify the *humanitarian category*. 3. Identify the *event location label* (from the allowed list). 4. Identify the *event year* (from the allowed list).  Below are the allowed label sets (you MUST choose only from these):  Allowed Disaster Types: Cyclone, Earthquake, Flood, Hurricane, Wildfire, earthquake, floods, hurricane, wildfire  Allowed Humanitarian Categories: caution_and_advice, displaced_people_and_evacuations, infrastructure_and_utility_da ...